# Dose contours

RADDOSE-3D writes `DoseState.R`, an R script that draws the dose at the end
of the run as nested surfaces -- everything above 0.1 MGy, above 20 and
above 30 -- and saves five views of them. It needs R, with its `rgl` and
`misc3d` packages. `pyraddose_3d.plotting` draws the same surfaces with
matplotlib, which the `plot` extra installs.

This notebook runs the insulin example from `inputs/` and draws it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from pyraddose_3d.models.experiment import Experiment
from pyraddose_3d.solve import simulate

experiment = Experiment.from_file(Path("inputs") / "insulin.txt")
simulation = simulate(experiment)

## As DoseState.R draws it

A surface where the dose crosses each of the R script's levels, in its
colours, in a box marking the crystal. The rotation axis, y, points up, and
the beam runs along z.

In [ ]:
from pyraddose_3d.plotting import plot_dose_contours

plot_dose_contours(simulation)
plt.show()

The crystal is turned through 180 degrees, so every voxel is in the beam at
some point and the palest surface wraps the whole crystal. The beam is 20 um
wide in x and 70 um tall in y, so the dose above 20 MGy is a lens, tall
along the rotation axis.

The surfaces are faceted because the voxels are 10 um apart: the example's
`PixelsPerMicron` is 0.1. A finer grid draws them smoother, and takes a
couple of seconds to run:

In [ ]:
finer = experiment.model_copy(
    update={"crystal": experiment.crystal.model_copy(update={"pixels_per_micron": 0.3})}
)
fine = simulate(finer)
plot_dose_contours(fine)
plt.show()

## Other levels

The R script's levels suit a crystal that reaches about 30 MGy. Give a
crystal its own: here a quarter, a half and three quarters of its highest
dose, in colours of our choosing. `DoseField` is the dose on the voxel grid,
zero outside the crystal.

In [ ]:
from pyraddose_3d.plotting import DoseField

field = DoseField.from_simulation(fine)
peak = field.dose.max()
print(f"highest dose {peak:.1f} MGy")

plot_dose_contours(
    field,
    levels=[round(fraction * peak) for fraction in (0.25, 0.5, 0.75)],
    colours=("gold", "darkorange", "darkred"),
    opacities=(0.15, 0.35, 1.0),
)
plt.show()

## The five views

`DoseState.R` saves five pictures: looking along the beam, then from higher
and further round. `view` takes the same `(theta, phi)`: theta round the
rotation axis from looking along the beam, and phi up from the horizontal.

In [ ]:
from pyraddose_3d.plotting import VIEWS

figure = plt.figure(figsize=(16, 3.6))
for number, (theta, phi) in enumerate(VIEWS, start=1):
    ax = figure.add_subplot(1, len(VIEWS), number, projection="3d")
    plot_dose_contours(field, view=(theta, phi), ax=ax)
    ax.get_legend().remove()
    ax.set_title(f"theta {theta}, phi {phi}")
plt.show()

`save_dose_contours` writes them as PNG files, as the R script does:
`output-DoseContours_0_0.png` and so on.

## From a finished run

`run_dose.py` writes `output-DoseState.csv`: every voxel's dose at the end
of the run. `DoseField.read_dose_state` reads it back, so a run can be drawn
again, with other levels, without running it again:

In [ ]:
import tempfile

from pyraddose_3d.outputs import write_outputs
from pyraddose_3d.plotting import save_dose_contours

directory = Path(tempfile.mkdtemp())
write_outputs(fine, prefix=str(directory / "output-"))

written = DoseField.read_dose_state(directory / "output-DoseState.csv")
for path in save_dose_contours(written, prefix=str(directory / "output-")):
    print(path.name)

`programs/dose/plot_dose_contours.py` does the same from the command line.
After `run_dose.py`, from the same directory:

```sh
python programs/dose/plot_dose_contours.py output-DoseState.csv
```

It saves the five views, and opens a window in which the contours can be
turned with the mouse. Its levels, colours and views are at the top of the
script.